PREPARANDO EL ESPACIO

In [ ]:
import os
import xml.etree.ElementTree as ET
from lxml import etree
import time
import google.generativeai as genai #llama a la api de gemini
from tqdm.notebook import tqdm #para la barra de progreso

In [ ]:
# --- SETTINGS ---

#Configuración de Gemini
with open('gemini.txt', 'r') as f: # recuerda colocar tu clave de  API en el archivo gemini.txt
    API_KEY = f.read()
genai.configure(api_key=API_KEY)
model = genai.GenerativeModel(
    model_name="gemini-2.5-flash",  # selecciona tu modelo de preferencia, puedes ver los que estan disponibles aquí: https://ai.google.dev/gemini-api/docs/models
    system_instruction=(
        "Eres un Senior Data Engineer. Tu tarea es documentar flujos SSIS en TEXTO PLANO. "
        "REGLA CRÍTICA: El orden del XML no es el orden de ejecución. Debes analizar los "
        "'DTS:PrecedenceConstraint' para reconstruir el flujo lógico paso a paso. "
        "Ignora componentes ausentes o deshabilitados. Usa la plantilla enviada."
    )
)

#Ruta: recuerda cambiar las rutas de las carpetas
# donde estarán tus paquetes ssis junto con tu archivo de parametros y tus archivos conmgr
# y la ruta donde deseas que se carguen los archivos de documentación.
ssis_location = "[RUTA_DE_TUS_DTSX_PARAMS_Y_CONMGRS]"
temp_output = "[RUTA_TEMPORAL_PARA_GUARDAR_LOS_RESUMENES_DE_TU_PARAM_Y_TUS_CONMGRS]"
docutentation_output_location = "[RUTA_DONDE_SE_DEPOSITARA_LA_DOCUMENTACIÓN_DE_LOS_DTSX]"

ruta_docu_conmgr = temp_output+"conmgr_summary.txt"
ruta_docu_params = temp_output+"params_summary.txt"

PREPARANDO CLASES Y FUNCIONES

In [ ]:
#CLASE CON LAS FUNCIONES PARA PROCESAR EL ARCHIVO Project.params Y LOS .conmgr

class SSISParser:
    def __init__(self, project_path):
        self.project_path = project_path
        self.ns = {'DTS': 'www.microsoft.com/SqlServer/Dts'}
        self.namespaces = {
            'SSIS': 'www.microsoft.com/SqlServer/SSIS',
            'DTS': 'www.microsoft.com/SqlServer/Dts'
        }

    def parse_conmgr(self):
        output = []
        output.append("-" * 60)
        output.append("--- CONEXIONES (.conmgr) ---")
        output.append("-" * 60)
        
        for file in os.listdir(self.project_path):
            if file.endswith(".conmgr"):
                try:
                    tree = ET.parse(os.path.join(self.project_path, file))
                    root = tree.getroot()

                    name = root.get(f"{{{self.ns['DTS']}}}ObjectName")
                    
                    expr_node = root.find(".//DTS:PropertyExpression[@DTS:Name='ConnectionString']", self.ns)
                    expression = expr_node.text.strip() if expr_node is not None else "Sin expresión dinámica"

                    obj_data_conn = root.find(".//DTS:ObjectData/DTS:ConnectionManager", self.ns)
                    static_conn = "No encontrada"
                    if obj_data_conn is not None:
                        static_conn = obj_data_conn.get(f"{{{self.ns['DTS']}}}ConnectionString")

                    output.append(f"Fichero: {file}")
                    output.append(f"Nombre Objeto: {name}")
                    output.append(f"Expresión (Lógica): {expression}")
                    output.append(f"Valor Estático: {static_conn}")
                    output.append("-" * 30)
                
                except Exception as e:
                    output.append(f"Error procesando {file}: {e}")
        
        return "\n".join(output)

    def parse_params(self):
        output = []
        output.append("--- PARÁMETROS DEL PROYECTO ---")
        param_file = os.path.join(self.project_path, "Project.params")
        
        if os.path.exists(param_file):
            tree = ET.parse(param_file)
            root = tree.getroot()
            for param in root.findall(".//SSIS:Parameter", self.namespaces):
                name = param.get("{www.microsoft.com/SqlServer/SSIS}Name")
                value_node = param.find("./SSIS:Properties/SSIS:Property[@SSIS:Name='Value']", self.namespaces)
                value = value_node.text if value_node is not None else "NULL/Vacio"
                output.append(f"Parámetro: {name} | Valor inicial: {value}")
        else:
            output.append("Archivo Project.params no encontrado.")
            
        return "\n".join(output)

parser = SSISParser(ssis_location)

In [ ]:
#FUNCION QUE LIMPIA EL XML DE LOS DTSX PARA REDUCIRLO EN TAMAÑO Y "RUIDO"
def clean_dtsx(input_path):
    ns = {'DTS': 'www.microsoft.com/SqlServer/Dts'}
    parser = etree.XMLParser(remove_blank_text=True)
    tree = etree.parse(input_path, parser)
    root = tree.getroot()

    # 1. Eliminar Metadata Visual (90% del peso)
    for node in root.xpath("//DTS:DesignTimeProperties", namespaces=ns):
        node.getparent().remove(node)

    # 2. Eliminar Tareas Deshabilitadas
    # Buscamos el atributo Disabled="True" en cualquier nivel
    for node in root.xpath("//*[@DTS:Disabled='True']", namespaces=ns):
        node.getparent().remove(node)

    return etree.tostring(tree, encoding='unicode', pretty_print=False)

In [ ]:
#FUNCION PARA DOCUMENTAR LOS PAQUETES SSIS (DTSX)
def document_all_packages(folder_path, params_ref, conmgr_ref):
    # Cargar los resúmenes de texto (los que ya tienes)
    with open(params_ref, 'r', encoding="utf-8") as f: p_text = f.read()
    with open(conmgr_ref, 'r', encoding="utf-8") as f: c_text = f.read()

    # Prompt que se mandará a la IA
    prompt = f"""
    Documenta el archivo txt adjunto, el cual contiene un codigo limpiado de un archivo tipo dtsx.
    
    CONTEXTO GLOBAL:
    PARAMETROS: {p_text}
    CONEXIONES: {c_text}
    
    INSTRUCCION: Reconstruye el orden de ejecución usando las restricciones de precedencia.
    Si una tarea no tiene entrada, es el inicio. Sigue las flechas (Constraints) hasta el final.

    Instrucciones de Formato:
    1. NO incluyas saludos, introducciones ni texto de relleno ("Aquí está tu análisis...").
    2. NO uses bloques de código (```markdown, ```text, etc). Devuelve texto plano directo.
    3. Sigue estrictamente la plantilla proporcionada abajo.
    4. Si un campo no aplica, escribe "N/A".

    Instrucciones de Análisis:
    - 1. RESUMEN DEL PROCESO: Describir qué hace basándose en la lógica.
    - 2. INFRAESTRUCTURA Y CONECTIVIDAD (ACTIVA): Identifica solo conexiones usadas por tareas habilitadas.
    - 3. PARAMETROS DEL PROYECTO RELACIONADOS: Lista los parámetros que el paquete usa, indica el nombre, tipo de dato y explica brevemente su uso. Si detectas que no se usa en paquete, márcalo como "(NO UTILIZADO)".
    - 4. FLUJO DE CONTROL (SOLO TAREAS HABILITADAS): Describe cada tarea activa en el Control Flow. ASEGURATE de solamente documentar tareas que esten HABILITADAS/ACTIVAS.
    - 5. FLUJO DE DATOS (DATA FLOW LOGIC): Describe cada Data Flow Task. ASEGURATE de solamente documentar flujos que esten HABILITADOS/ACTIVOS.
    
    Utiliza la siguiente plantilla al pie de la letra:

    --- PLANTILLA DE INICIO ---
    FICHA TECNICA DE PROCESO ETL NOMBRE DEL PAQUETE: [Nombre]
    
    1. RESUMEN DEL PROCESO:

    
    2. INFRAESTRUCTURA Y CONECTIVIDAD (ACTIVA)
    a) Conexión 1: [Nombre]
    -Tipo: [SQL Server / Excel / etc]
    -Servidor/Ruta: [Nombre o ruta]
    -Base de Datos: [Valor resuelto]
    
    b) Conexión 2: [Nombre]
    -Tipo: [SQL Server / Excel / etc]
    -Servidor/Ruta: [Nombre o ruta]
    -Base de Datos: [Valor resuelto]

    c) ...

    
    3. PARAMETROS DEL PROYECTO RELACIONADOS
    - Parametro1: Explicación...
    - Parametro2: Explicación... (NO UTILIZADO)

    
    4. FLUJO DE CONTROL (SOLO TAREAS HABILITADAS):    
    a) TAREA: [Nombre]
    - TIPO: [Execute SQL Task / Data Flow / etc]
    - LOGICA: [Si es SQL, poner el código aquí. Si es otra, describir acción]
    - PRECEDENCIA: [Cual tarea o flujo de datos se ejecutó previamente, si no hay precedencia coloca "NA"]

    b) TAREA: [Nombre]
    - TIPO: [Execute SQL Task / Data Flow / etc]            
    - LOGICA: [Si es SQL, poner el código aquí. Si es otra, describir acción]
    - PRECEDENCIA: [Cual tarea o flujo de datos se ejecutó previamente]

    c)...

    
    5. FLUJO DE DATOS (DATA FLOW LOGIC):
    - ORIGEN: [Tabla o Query]            
    - TRANSFORMACIONES: [Mapeos, Derived Columns, Lookups activos. Considera el orden de pasos]
    - FILTROS: Indica los filtros más utilizados a los largo del proceso, es decir lo que se coloca en los 'where' de los select. Estos suelen representar reglas de negocio.
    - DESTINO: [Tabla final y base de datos]
    - PRECEDENCIA: [Cual tarea o flujo de datos se ejecutó previamente]

    """

    for file in tqdm(paquetes):
        path = os.path.join(folder_path, file)
        
        # Limpiar XML
        clean_xml = clean_dtsx(path)
        temp_clean_path = f"{docutentation_output_location}//temp//temp_{file}.txt"
        with open(temp_clean_path, "w", encoding="utf-8") as f:
            f.write(clean_xml)

        # Subir a la API de Archivos
        sample_file = genai.upload_file(path=temp_clean_path)
        while sample_file.state.name == "PROCESSING":
            time.sleep(2)
            sample_file = genai.get_file(sample_file.name)

        try:
            response = model.generate_content([sample_file, prompt])            
            # Guardar resultado
            output_name = docutentation_output_location+"//DOCU__"+file.replace(".dtsx", ".txt")
            with open(output_name, "w", encoding="utf-8") as f:
                f.write(response.text)
        
        except Exception as e:
            print(f"Error crítico en el paquete {file}: {e}")
            # continue # Salta al siguiente paquete
        
        finally:# Esto se ejecuta SIEMPRE, haya error o no
            # Limpieza
            if os.path.exists(temp_clean_path):
                os.remove(temp_clean_path)
            try:
                genai.delete_file(sample_file.name)
            except:
                pass

PROCESAR LOS ARCHIVOS

In [ ]:
#PROCESAR Y RESUMIR LOS ARCHIVOS .conmgr
conmgr_docu = parser.parse_conmgr()
with open(ruta_docu_conmgr, "w", encoding="utf-8") as new_archivo:
    new_archivo.write(conmgr_docu)
print(conmgr_docu)

In [ ]:
#PROCESAR Y RESUMIR EL ARCHIVO Project.params PARA DESPUES DEJARLO EN UN TXT
params_docu = parser.parse_params()
with open(ruta_docu_params, "w", encoding="utf-8") as new_archivo:
    new_archivo.write(params_docu)
print(params_docu)

In [ ]:
#PROCESAR Y RESUMIR EL ARCHIVO Project.params PARA DESPUES DEJARLO EN UN TXT
paquetes = [f for f in os.listdir(ssis_location) if f.endswith('.dtsx')]
paquetes = paquetes[:3]
print("#paquetes: "+str(len(paquetes)))
paquetes

In [ ]:
# DOCUMENTAR
document_all_packages(ssis_location, ruta_docu_params, ruta_docu_conmgr)